*Tre missionari e Tre cannibali* devono attraversare un fiume.

- Missionari e cannibali sono inizialmente sulla sponda sinistra del fiume e devono raggiungere la sponda destra.
- Per farlo, hanno a disposizione una barca che può contenere al massimo due persone.
- La barca non può attraversare il fiume senza nessuno a bordo.
- Il numero di cannibali non può mai essere superiore al numero di missionari su una delle due sponde.

L'**obiettivo** à quello di trovare una **sequenza minima** di AZIONI che permetta a tutti di attraversare il fiume




In [ ]:
class Agent():

    def __init__(self):
        pass


    # Questo metodo è definito come statico perché 
    # non ha bisogno di accedere a nessuna variabile della classe.
    @staticmethod
    def apply_move(state,move):
        pass # Applica una mossa ad uno stato

    def next_states(self,state):
        pass # Genera tutti gli stati raggiungibili da uno stato
    
    def is_goal(self,state):
        pass # Verifica se uno stato sia lo stato finale

    def bfs(self):
        pass # implementa la ricerca in ampiezza
    
        

### Rappresentazione dello stato

Il primo problema da porsi in un problema di ricerca e' come rappresentare lo stato:
- lo stato deve contenere tutte le informazioni necessarie per verficare i **vincoli del problema**
- I dettagli irrilevanti per il problema non devono essere memorizzati nello stato
- La rappresentazione deve (idealmente) essere efficiente da manipolare(se controllare la validita' di uno stato e' costoso allora l'algoritmo sara' costoso)


In [ ]:
state = {
    'L' : {'m' : 3, 'c': 3},
    'R' : {'m': 0, 'c' : 0},
    'boat' : 'L'
}

goal = {
    'L' : {'m' : 0, 'c':0},
    'R' : {'m': 3, 'c' : 3},
    'boat' : 'R'
}

### Movimenti possibili
Definiamo quindi una lista di azioni possibili che, applicate ad uno stato, generano un nuovo stato. Necessitiamo di una funzione che ci permetta di verificare se uno stato è valido secondo le regole del problema per rimuovere gli stati non validi dalla frontiera. Uno stato non è valido se su una delle due sponde sono presenti missionari in minoranza rispetto ai cannibali (o se il numero di missionari o cannibali è negativo).

In [ ]:
moves = {{'m':1, 'c':0}, #spostare un missionario
         {'m':0, 'c':1}, # spostare un cannibale
         {'m':1, 'c':1}, #spostare un missionario e un cannbale
         {'m':0, 'c':2}, # spostare due cannibali
         {'m':2, 'c':0}} # spostare due missionari

In [ ]:
def is_valid(state):
    for side in['L', "R"]:
        if state[side]['m'] < 0 or state[side]['c'] < 0:
            return False
        if state[side]['c'] > state[side]['m']:
            return False
    return True

### Applicare una mossa

Dato uno stato e una mossa, dobbiamo generare un nuovo stato. Per prima cosa copiamo lo stato corrente. Cambiamo la posizione della barca, e rimuoviamo le persone dalla sponda di partenza e le aggiungiamo alla sponda di arrivo.

In [ ]:
import copy

@staticmethod
def apply_move(state,move):
    new_state = copy.deepcopy(state) # Copiamo lo stato attuale
    new_state['boat'] = 'L' if state['boat']=='R' else 'R'
    for person in ['m','c']:
        # Togliamo le persone dalla sponda SORGENTE
        new_state[state['boat']][person] -= move[person]
        new_state[new_state['boat']][person] += move[person]
    return new_state


### Generazione degli stati successivi
Creiamo nuovi stati prendendo in considerazione tutte le azioni possibili. Solamente le azioni che non violano le regole del problema verranno restituite

In [ ]:
def next_states(self, state):
    def is_valid(state):
        for side in ['L','R']:
            if state[side]['c'] < 0 or state[side]['m'] < 0:
                return False
            if state[side]['c'] > state[side]['m']:
                return False
        return True

    for move in self.moves:
        new_state = self.apply_move(state,move)
        if is_valid(state):
            yield new_state

### Ricerca in ampiezza
A questo punto, possiamo eseguire una ricerca in ampiezza per trovare una sequenza di stati che porti dallo stato iniziale a quello finale.
Il livello di astrazione è sufficientemente alto da poter riutilizzare il codice della ricerca in ampiezza che abbiamo visto in precedenza.

Gli algoritmi di ricerca sono algoritmi generici che possono essere riutilizzati in molti problemi diversi una volta che si è in grado di modellare il problema
col giusto livello di astrazione.

In [ ]:
start = {
    'L' : {'m' : 3, 'c' :3},
    'R': {'m' : 0, 'c' :0},
    'boat':'L'
}

goal = {
    'L' : {'m' : 0, 'c' :0},
    'R': {'m' : 3, 'c' :3},
    'boat':'R'
}

moves = [
    {'m': 1, 'c': 0},#spostare un missionario
    {'m': 0, 'c': 1},# spostare un cannibale
    {'m': 1, 'c': 1},#spostare un missionario e un cannbale
    {'m': 0, 'c': 2},# spostare due cannibali
    {'m': 2, 'c': 0},# spostare due missionari
]




import copy

class Agent():
    def __init__(self, start = start, goal=goal, moves=moves):
        self.moves = moves
        self.state = start
        self.goal=goal
        self.frontier = [[start]]

    @staticmethod
    # applica una mossa ad uno stato
    def apply_move(state,move):
        new_state = copy.deepcopy(state)
        new_state['boat'] = 'L' if state['boat'] == 'R' else 'R'

        for person in ['c','m']:
            new_state[state['boat']][person] -= move[person]
            new_state[new_state['boat']][person] += move[person]
        return new_state

    def next_states(self,state):
        # genera tutti gli stati raggiungibili da uno stato
        def is_valid(state):
            for side in ['L','R']:
                if state[side]['c'] < 0 or state[side]['m'] < 0:
                    return False
                if state[side]['m'] > 0 and state[side]['c'] > state[side]['m']:
                    return False
            return True

        for move in self.moves:
            new_state = self.apply_move(state, move)
            if is_valid(new_state):
                yield new_state

    def is_goal(self,path):
        return path[-1] == self.goal

    def bfs(self):
        while self.frontier:
            path = self.frontier.pop(0)
            if self.is_goal(path):
                return path
            for s in self.next_states(path[-1]):
                if s not in path:
                    self.frontier.append(path + [s])

if __name__ == "__main__":
    agent = Agent()
    soluzione = agent.bfs()
    print(f"Soluzione trovata in {len(soluzione) - 1} passaggi:")
    for step, s in enumerate(soluzione):
        print(f"Passo {step}: Barca su {s['boat']} | SX: {s['L']} | DX: {s['R']}")

Soluzione trovata in 11 passaggi:
Passo 0: Barca su L | SX: {'m': 3, 'c': 3} | DX: {'m': 0, 'c': 0}
Passo 1: Barca su R | SX: {'m': 2, 'c': 2} | DX: {'m': 1, 'c': 1}
Passo 2: Barca su L | SX: {'m': 3, 'c': 2} | DX: {'m': 0, 'c': 1}
Passo 3: Barca su R | SX: {'m': 3, 'c': 0} | DX: {'m': 0, 'c': 3}
Passo 4: Barca su L | SX: {'m': 3, 'c': 1} | DX: {'m': 0, 'c': 2}
Passo 5: Barca su R | SX: {'m': 1, 'c': 1} | DX: {'m': 2, 'c': 2}
Passo 6: Barca su L | SX: {'m': 2, 'c': 2} | DX: {'m': 1, 'c': 1}
Passo 7: Barca su R | SX: {'m': 0, 'c': 2} | DX: {'m': 3, 'c': 1}
Passo 8: Barca su L | SX: {'m': 0, 'c': 3} | DX: {'m': 3, 'c': 0}
Passo 9: Barca su R | SX: {'m': 0, 'c': 1} | DX: {'m': 3, 'c': 2}
Passo 10: Barca su L | SX: {'m': 1, 'c': 1} | DX: {'m': 2, 'c': 2}
Passo 11: Barca su R | SX: {'m': 0, 'c': 0} | DX: {'m': 3, 'c': 3}
